# Watching the trained agents play — qualitative animations

**Direction:** `endogenous-action-interactive-world` · 2026-07-29. Purpose: *see* what each trained agent actually
does in the world, in exactly the view you get from `python scripts/play.py`, plus one extra panel showing what the
model **predicts** it will see next.

Each animation is generated by the **same `Emulator` class that `scripts/play.py` uses** (imported, not
re-implemented), so the 2D world panel, the keyboard overlay, the white per-object force vectors, the observation
waterfall and the status readout are identical to the interactive tool. The model's discrete actions are mapped
onto the same keys a human would press. **World settings are read from each checkpoint** so the visualised world
matches the one the model was trained in (dynamics, death-on-collision, death-on-wall, death noise frames,
initial momentum, observation noise).

**Panels:** 2D world (latent state) · keys pressed (the model's action) · status · the real 1D observation
waterfall · **the model's predicted next observation** (and, in one case, the observer's prediction beside it).

GIFs are written to `runs/endogenous/animations/` (paths printed below each one) so they are easy to save/copy.

> **⚠ Known deviation from the repo standard.** These runs used **`obs_noise_std = 0.05`**, whereas every prior
> dataset in this project (0–8, including dataset 4 used by the exogenous-action work) uses **0.2**. The value
> leaked in from a `scripts/play.py` display default and was not a deliberate experimental choice. All endogenous
> runs share it, so comparisons *within* this thread are valid, but **absolute RMSE / probe-R² values are not
> comparable to the earlier notebooks** (less noise ⇒ easier prediction and easier probing). The training script
> default is now back to 0.2; a matched re-run is queued.

## Run definitions (copied from the canonical registry `ENDOGENOUS_RUNS.md`)

| code | descriptive label | level / world | goal | architecture | training | seed |
|---|---|---|---|---|---|---|
| `L1` | L1 shift · prediction-only · 256h | `shift` dynamics (action = position delta, guarded; **death impossible**) | none | 256 hidden, Linear enc/dec | 2 500 it | 0 |
| `L2` | L2 force · prediction-only · 256h | `force` dynamics, death on collision **and** wall | none | 256 hidden, Linear enc/dec | 2 500 it | 0 |
| `L3` | L3 force+goal · **weak** · 256h · seed 0 | `force`, lethal | **survive** (REINFORCE) | 256 hidden, Linear enc/dec | 6 000 it | 0 |
| `L3b` | L3 force+goal · weak · 256h · **seed 1** | `force`, lethal | survive | 256 hidden, Linear enc/dec | 6 000 it | 1 |
| `L3s0` | L3 force+goal · **strong** · 512h · seed 0 | `force`, lethal | survive | **512 hidden, 2-layer MLP encoder + residual MLP decoder** | 25 000 it, 5-step free-run loss | 0 |
| `L3s1` | L3 force+goal · strong · 512h · **seed 1** | `force`, lethal | survive | strong (as above) | 25 000 it | 1 |
| `L2s0` | L2 force · prediction-only · **strong** | `force`, lethal | none (**capacity control**) | strong | 12 000 it | 0 |
| `L3s0_ckpt` | L3 strong seed 0 **with intermediate checkpoints** | `force`, lethal | survive | strong | 25 000 it, saved every 2 500 | 0 |

**Roles.** *actor* = the model whose policy head emits the action applied to the world (trained on prediction + —
at level 3 — REINFORCE into the shared trunk). *observer* = an identical network trained on the **same**
observations and **fed the actor's actions**, but which never acts. Suffix key: no suffix / `b` = the original
("weak") config differing only in seed; `s` = the strong config; trailing digit = seed.

In [ ]:
# [1] Setup — import the SAME emulator that scripts/play.py uses.
import sys, argparse
from pathlib import Path
sys.path.insert(0, '../../../..')
import torch
from IPython.display import display, Markdown, Image
from scripts.play import Emulator, ModelDriver, build_world
from scripts.eval_editability_endogenous import load as load_ckpt
from pim.world_models.actor_gru import EndogenousActorConfig, EndogenousActorGRU

ROOT = Path('../../../../runs/endogenous')
OUT = ROOT / 'animations'; OUT.mkdir(parents=True, exist_ok=True)
FRAMES, FPS, DPI = 100, 10, 55   # keeps each GIF ~4 MB so the notebook stays openable

def world_for(ck, seed=3):
    """Build a world whose settings MATCH the ones this checkpoint was trained in."""
    ic, sim = ck['interactive_cfg'], ck['sim_cfg']
    return build_world(argparse.Namespace(
        dynamics=ic['dynamics'], n_objects=2, seed=seed, obs_res=sim['obs_res'],
        obs_noise=sim['obs_noise_std'], death_on_collision=ic['death_on_collision'],
        death_on_wall=ic['death_on_wall'], reset_noise_frames=ic['reset_noise_frames']))

def animate(tag, title, ckpt='ckpt_final.pt', predictors=None, seed=3, fname=None):
    """Render one play.py-style animation for a checkpoint; returns the GIF path."""
    actor, observer, sim, ck = load_ckpt(ROOT / tag / ckpt)
    w = world_for(ck, seed=seed)
    preds = predictors(actor, observer) if callable(predictors) else [("model's predicted next obs", actor)]
    emu = Emulator(w, driver=ModelDriver(actor), predictors=preds)
    emu.title = title
    path = OUT / (fname or f'{tag}.gif')
    emu.save(str(path), frames=FRAMES, fps=FPS, dpi=DPI)
    ic = ck['interactive_cfg']
    display(Markdown(f"**{title}**  \n`{path.resolve()}`  \n"
                     f"world: dynamics=`{ic['dynamics']}`, death-on-collision=`{ic['death_on_collision']}`, "
                     f"death-on-wall=`{ic['death_on_wall']}`, death-noise-frames=`{ic['reset_noise_frames']}`, "
                     f"obs noise σ=`{ck['sim_cfg']['obs_noise_std']}`"))
    display(Image(filename=str(path)))
    return path
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. One animation per trained agent (actor drives; its own prediction beside the truth)

Read the middle waterfall as *what really happened* and the right waterfall as *what the model expected*. Where the
two diverge, the world model is wrong. Note the level-1 world is **guarded** (no deaths possible), while levels 2
and 3 are lethal — at level 2 the agent has **no goal**, so it dies constantly; at level 3 it has learned to survive.

In [ ]:
# [L1] animation
animate('L1', 'L1 shift · prediction-only · 256h · seed 0 — no goal, guarded world (death impossible)')

In [ ]:
# [L2] animation
animate('L2', 'L2 force · prediction-only · 256h · seed 0 — lethal world, NO goal (dies constantly)')

In [ ]:
# [L3] animation
animate('L3', 'L3 force+goal · weak · 256h · seed 0 — trained to survive (REINFORCE)')

In [ ]:
# [L2s0] animation
animate('L2s0', 'L2 force · prediction-only · strong · seed 0 — capacity control, still NO goal')

In [ ]:
# [L3s0] animation
animate('L3s0', 'L3 force+goal · strong · 512h · seed 0 — trained to survive')

In [ ]:
# [L3s1] animation
animate('L3s1', 'L3 force+goal · strong · 512h · seed 1 — seed replication')

## 2. Actor vs observer — same world, same actions, two different predictors

The **actor** drives the world. The **observer** (identical architecture, never acts) watches the *same*
observation stream and is told the actor's action, then predicts the next observation. Comparing the two right-hand
waterfalls shows whether *acting* changes what the model predicts.

In [ ]:
animate('L3s0',
        'L3 force+goal · strong · seed 0 — actor drives; ACTOR vs OBSERVER predictions',
        predictors=lambda a, o: [("ACTOR's predicted next obs", a), ("OBSERVER's predicted next obs", o)],
        fname='L3s0_actor_vs_observer.gif')

## 3. The same agent across training — untrained → partway → trained

All three are the **same run** (`L3s0_ckpt`: L3 force+goal, strong config, seed 0), at different training
checkpoints. Watch both the *behaviour* (does it avoid walls and the other object?) and the *prediction* panel
(does the predicted waterfall track the real one?) improve together.

In [ ]:
# [stages] untrained → partway → trained. Uses whichever intermediate checkpoints exist.
import re
ck_dir = ROOT / 'L3s0_ckpt'
its = sorted(int(re.search(r'it(\d+)', p.name).group(1)) for p in ck_dir.glob('ckpt_it*.pt')) if ck_dir.exists() else []
print('intermediate checkpoints:', its)
stages = []
if its:
    final = ck_dir / 'ckpt_final.pt'
    last = ('ckpt_final.pt', 'fully trained (iteration 25000)') if final.exists() \
           else (f'ckpt_it{its[-1]}.pt', f'most trained available (iteration {its[-1]})')
    stages = [(f'ckpt_it{its[0]}.pt', f'barely trained (iteration {its[0]})'),
              (f'ckpt_it{its[len(its)//2]}.pt', f'partway trained (iteration {its[len(its)//2]})'),
              last]
else:
    print('no intermediate checkpoints yet — run scripts/train_endogenous.py with --ckpt-every')
for ckpt, stage in stages:
    animate('L3s0_ckpt', f'L3 force+goal · strong · seed 0 — {stage}', ckpt=ckpt,
            fname=f'L3s0_stage_{ckpt.replace(".pt","")}.gif')

## 4. Autoregressive (closed-loop) operation — the model runs on its OWN predictions

Everything above is **teacher-forced**: the model sees the *real* observation every step, so the prediction panel
is only ever a **one-step-ahead** prediction. That flatters it.

Here the driver is switched to **closed loop**: after a 15-frame warm-up the model **stops seeing the world** and
consumes its **own predicted observation** each step, while the actions it chooses from that imagined state are
still applied to the real world. Prediction error now compounds, so watch for two failures together:

1. the **imagined waterfall drifts away** from the real one (the bands blur, bend, or freeze), and
2. the **actions start to fail** — the real 2D world shows collisions and deaths, because the policy is now acting
   on a hallucinated state.

This is the honest measure of world-model quality, and it is also exactly the regime the editability rollouts and
the action-channel test in `endogenous_grabbability.ipynb` operate in — which is why numbers there look much worse
than the teacher-forced panels above would suggest.

In [ ]:
# [autoregressive] the model dreams: after warm-up it consumes only its own predictions.
from scripts.play import AutoregressiveModelDriver

def animate_autoregressive(tag, title, warmup=15, seed=3):
    actor, observer, sim, ck = load_ckpt(ROOT / tag / 'ckpt_final.pt')
    w = world_for(ck, seed=seed)
    drv = AutoregressiveModelDriver(actor, warmup=warmup)
    emu = Emulator(w, driver=drv,
                   predictors=[(f'model IMAGINED obs (closed loop after {warmup} frames)', 'driver')])
    emu.title = title
    path = OUT / f'{tag}_autoregressive.gif'
    emu.save(str(path), frames=FRAMES, fps=FPS, dpi=DPI)
    display(Markdown(f'**{title}**  \n`{path.resolve()}`'))
    display(Image(filename=str(path)))
    return path

animate_autoregressive('L3', 'L3 force+goal · WEAK 256h · seed 0 — CLOSED LOOP (model runs on its own predictions)')
animate_autoregressive('L3s0', 'L3 force+goal · STRONG 512h · seed 0 — CLOSED LOOP (model runs on its own predictions)')

### 4b. The actor's dream vs the OBSERVER's dream — both closed loop, same actions

The actor drives and dreams (closed loop after the warm-up). The **observer** is run closed-loop too — it consumes
**its own** predictions, but is told the **same action sequence** the actor applied and gets the **same warm-up**.
So the three waterfalls are: **reality**, the **actor's** imagination, the **observer's** imagination, all aligned.

This is the sharpest available test of whether *acting* changes the world model's imagination. If the actor's dream
tracked reality better than the observer's, that would be direct evidence for the endogenous-action hypothesis.

In [ ]:
# [autoregressive pair] reality | actor's dream | observer's dream — all closed loop, same actions.
from scripts.play import AutoregressivePredictor

def animate_dream_pair(tag, title, warmup=15, seed=3):
    actor, observer, sim, ck = load_ckpt(ROOT / tag / 'ckpt_final.pt')
    w = world_for(ck, seed=seed)
    emu = Emulator(w, driver=AutoregressiveModelDriver(actor, warmup=warmup),
                   predictors=[('ACTOR imagined (closed loop)', 'driver'),
                               ('OBSERVER imagined (closed loop, same actions)',
                                AutoregressivePredictor(observer, warmup=warmup))])
    emu.title = title
    path = OUT / f'{tag}_autoregressive_actor_vs_observer.gif'
    emu.save(str(path), frames=FRAMES, fps=FPS, dpi=DPI)
    display(Markdown(f'**{title}**  \n`{path.resolve()}`  \n'
                     f'real-world deaths during these {FRAMES} frames: **{w.deaths}**'))
    display(Image(filename=str(path)))
    return path

animate_dream_pair('L3', 'L3 force+goal · WEAK 256h · seed 0 — CLOSED LOOP: reality vs actor dream vs observer dream')
animate_dream_pair('L3s0', 'L3 force+goal · STRONG 512h · seed 0 — CLOSED LOOP: reality vs actor dream vs observer dream')

In [ ]:
# [autoregressive-quantified] the same comparison as a number: death rate when the model sees the
#   real world each step, vs when it runs on its own predictions.
from scripts.play import ModelDriver as _MD, AutoregressiveModelDriver as _AD
rows = ['| model | control mode | deaths per 1000 frames |', '|---|---|---|']
for tag, lab in [('L3', 'L3 goal · weak'), ('L3s0', 'L3 goal · strong')]:
    actor, observer, sim, ck = load_ckpt(ROOT / tag / 'ckpt_final.pt')
    ic = ck['interactive_cfg']
    for mode, mk in [('teacher-forced (sees the real observation)', lambda m: _MD(m)),
                     ('CLOSED LOOP (its own predictions)', lambda m: _AD(m, warmup=15))]:
        deaths = 0
        for seed in range(6):
            w = world_for(ck, seed=seed); drv = mk(actor)
            for _ in range(300):
                w.step(drv.act(w))
            deaths += w.deaths
        rows.append(f'| {lab} | {mode} | **{1000*deaths/(6*300):.1f}** |')
display(Markdown('\n'.join(rows)))
display(Markdown('For reference, the **no-goal** control (`L2 force · strong`) dies at **79.1** per 1000 frames, '
                 'and the goal-trained actor at **0.33** during training. So acting inside its own imagination is '
                 'roughly as bad as having no policy at all.'))

## What to look for (interpretation)

- **Level 2 vs level 3** is the clearest behavioural contrast: with no goal the agent lets objects drift into walls
  and each other (frequent death → noise frames → rebirth in the waterfall); with the survival goal it actively
  holds them apart and away from the walls.
- **The prediction panel is visibly blurrier than the truth.** This is the quantified `sharpness` limitation from
  `endogenous_grabbability.ipynb`: the models predict the *band positions* well but smear their edges, and the
  stronger configuration did not fix it.
- **Closed loop is where it breaks.** Teacher-forced, the goal-trained actor dies ~2.8 times per 1000 frames; on
  its own predictions that becomes ~**86–88**, i.e. **≈31× worse** and about the same as the *no-goal* control. The
  policy depends on fresh observations; the imagined state degrades too fast to act on. This is the same regime the
  editability and action-channel tests operate in, and explains why their numbers look so much worse than the
  one-step-ahead panels above.
- **Both roles dream the SAME fantasy.** In the closed-loop pair animation the actor's and the observer's imagined
  streams are nearly indistinguishable — smooth, well-separated, death-free — while *reality* is full of collisions
  and rebirth noise. So the closed-loop failure is **architectural, not agency-related**, and it further weakens the
  actor-vs-observer contrast: if acting does not change what the model imagines, it is hard to argue it has
  restructured the latent.
- **Actor vs observer predictions look very similar** — consistent with the measured identifiability gap shrinking
  to near zero once both models are trained to strength.